In [10]:
import pandas as pd
import shutil
import os

CODE_DIR = '/kaggle/input/datasets/kaggle/meta-kaggle-code'

# 1. Load the Kernel Versions to get the file IDs
# (Assuming you already filtered Kernels.csv for highly-voted Python notebooks)
print("Loading the data...")
target_kernels = pd.read_csv('/kaggle/input/datasets/kaggle/meta-kaggle/Kernels.csv')
target_kernels = target_kernels[target_kernels['TotalVotes'] > 40].sample(n=5000, random_state=42)



# Join with KernelVersions to get the most recent Version ID for each notebook
versions = pd.read_csv('/kaggle/input/datasets/kaggle/meta-kaggle/KernelVersions.csv', usecols=['Id', 'ScriptId'])
targets = target_kernels.merge(versions, left_on='Id', right_on='ScriptId', how='inner')

# Keep only the latest version ID for each notebook
latest_versions = targets.groupby('ScriptId')['Id_y'].max().astype(str).tolist()

# 2. Setup output directory
os.makedirs('/kaggle/working/corpus', exist_ok=True)

# 3. Copy files directly from the mounted dataset
success_count = 0
print("Copying files...")
for version_id in latest_versions:
    v_int = int(version_id)
    
    # Millions chunk, zero-padded to 4 digits (e.g., 9001037 -> 0009)
    p1 = str(v_int // 1_000_000).zfill(4)
    
    # Thousands chunk, zero-padded to 3 digits (e.g., 9001037 -> 001)
    p2 = str((v_int // 1_000) % 1_000).zfill(3)
    
    file_path = os.path.join(CODE_DIR, p1, p2, f"{version_id}.ipynb")
    print(".", end="")
    if os.path.exists(file_path):
        shutil.copy(file_path, f'/kaggle/working/corpus/{version_id}.ipynb')
        success_count += 1

print("")
print(f"Successfully copied {success_count} notebooks.")

# 4. Zip the payload
shutil.make_archive('/kaggle/working/kaggle_corpus', 'zip', '/kaggle/working/corpus')
print("kaggle_corpus.zip is ready for download!")

Loading the data...
Copying files...
...................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [6]:
import pandas as pd
import os

META_DIR = '/kaggle/input/datasets/kaggle/meta-kaggle'

kernels = pd.read_csv(os.path.join(META_DIR, 'Kernels.csv'), nrows=5)
versions = pd.read_csv(os.path.join(META_DIR, 'KernelVersions.csv'), nrows=5)

print("Kernels columns:", list(kernels.columns))
print("KernelVersions columns:", list(versions.columns))

Kernels columns: ['Id', 'AuthorUserId', 'CurrentKernelVersionId', 'ForkParentKernelVersionId', 'ForumTopicId', 'FirstKernelVersionId', 'CreationDate', 'EvaluationDate', 'MadePublicDate', 'IsProjectLanguageTemplate', 'CurrentUrlSlug', 'Medal', 'MedalAwardDate', 'TotalViews', 'TotalComments', 'TotalVotes']
KernelVersions columns: ['Id', 'ScriptId', 'ParentScriptVersionId', 'ScriptLanguageId', 'AuthorUserId', 'CreationDate', 'VersionNumber', 'Title', 'EvaluationDate', 'IsChange', 'TotalLines', 'LinesInsertedFromPrevious', 'LinesChangedFromPrevious', 'LinesUnchangedFromPrevious', 'LinesInsertedFromFork', 'LinesDeletedFromFork', 'LinesChangedFromFork', 'LinesUnchangedFromFork', 'TotalVotes', 'IsInternetEnabled', 'RunningTimeInMilliseconds', 'AcceleratorTypeId', 'DockerImage']


In [7]:
high_vote_kernels = kernels[kernels['TotalVotes'] > 50]
python_versions = versions[versions['ScriptLanguageId'] == 9]
targets = high_vote_kernels.merge(python_versions, left_on='Id', right_on='ScriptId', how='inner')
latest_versions = targets.groupby('ScriptId')['Id_y'].max().astype(str).tolist()[:1000]

In [8]:
print("Total high-vote kernels:", len(high_vote_kernels))
print("Total python versions:", len(python_versions))
print("Merged targets count:", len(targets))
print("Unique latest versions targeted:", len(latest_versions))

Total high-vote kernels: 1
Total python versions: 4
Merged targets count: 0
Unique latest versions targeted: 0
